# Prova 01 – Modelos Financeiros Computacionais
**Aluno:** Guilherme Mendes Rosa  
**Disciplina:** MFC – 2026/S1  
**Data de entrega:** 07/07/2026  

**Ativos (Atividade 01 – Ana Paula Schultze):** PETR4, EMBJ3, WEGE3, ABEV3, BPAC11  
**Pesos Atividade 01:** 35%, 20%, 15%, 20%, 10%

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Ativos e pesos da Atividade 01
TICKERS      = ['PETR4.SA', 'EMBJ3.SA', 'WEGE3.SA', 'ABEV3.SA', 'BPAC11.SA']
NOMES        = ['PETR4', 'EMBJ3', 'WEGE3', 'ABEV3', 'BPAC11']
PESOS_ATIV01 = np.array([0.35, 0.20, 0.15, 0.20, 0.10])

In [ ]:
# Funções do exercício prático do professor

def calcula_retorno(carteira, retornos):
    """Calcula o retorno esperado de uma carteira."""
    return np.sum(carteira * retornos)

def calcula_risco(carteira, covariancia):
    """Calcula o risco de uma carteira."""
    return np.sqrt(np.dot(carteira.T, np.dot(covariancia, carteira)))

def calcula_minima_variancia(retornos, covariancia):
    """
    Calcula a carteira de mínima variância (solução analítica).
    Projeta os pesos no simplex positivo (sem vendas a descoberto).
    """
    num_ativos = len(retornos)
    ones = np.ones(num_ativos)
    cov_inv = np.linalg.inv(covariancia)
    w = np.dot(cov_inv, ones) / np.dot(ones, np.dot(cov_inv, ones))
    # Projeta no simplex positivo (sem shorts)
    w = np.clip(w, 0, None)
    w = w / w.sum()
    return w

## Questão 2 – Análise dos Últimos 36 Meses (março/2023 a março/2026)

### 2a) Carteira de Mínima Variância – Período Completo e Mês a Mês

In [ ]:
# Download dos preços: março/2023 a março/2026
START = '2023-03-01'
END   = '2026-04-01'  # exclusivo no yfinance, pega até 31/03/2026

raw = yf.download(TICKERS, start=START, end=END, progress=False, auto_adjust=True)
precos = raw['Close'].copy()
precos.columns = NOMES
precos = precos.dropna(how='all').ffill()

print(f'Período baixado: {precos.index[0].date()} → {precos.index[-1].date()}')
print(f'Dias úteis: {len(precos)}')
print(f'\nPrimeiros preços:')
print(precos.head(3).to_string())
print(f'\nÚltimos preços:')
print(precos.tail(3).to_string())

In [ ]:
# ── Carteira MV: PERÍODO COMPLETO (36 meses) ──

ret_diario = precos.pct_change().dropna()
ret_anual  = ret_diario.mean() * 252
cov_anual  = ret_diario.cov()  * 252

pesos_mv_total  = calcula_minima_variancia(ret_anual.values, cov_anual.values)
retorno_mv_total = calcula_retorno(pesos_mv_total, ret_anual.values)
risco_mv_total   = calcula_risco(pesos_mv_total, cov_anual.values)

print('=' * 50)
print('Carteira de Mínima Variância – Período Completo')
print('=' * 50)
for nome, w in zip(NOMES, pesos_mv_total):
    print(f'  {nome}: {w:.4f}  ({w*100:.2f}%)')
print(f'\nRetorno anualizado: {retorno_mv_total*100:.2f}%')
print(f'Risco anualizado:   {risco_mv_total*100:.2f}%')

In [ ]:
# ── Carteira MV: MÊS A MÊS ──

ret_diario_mensal = ret_diario.copy()
ret_diario_mensal['_mes'] = ret_diario_mensal.index.to_period('M')
meses = sorted(ret_diario_mensal['_mes'].unique())

resultados_mensais = []

for mes in meses:
    df_mes = ret_diario_mensal[ret_diario_mensal['_mes'] == mes].drop(columns='_mes')

    # Precisa de pelo menos 6 dias úteis para estimar covariância 5x5
    if len(df_mes) < 6:
        continue

    r_a = df_mes.mean() * 252
    c_a = df_mes.cov()  * 252

    try:
        w = calcula_minima_variancia(r_a.values, c_a.values)
    except np.linalg.LinAlgError:
        continue

    linha = {'Mês': str(mes)}
    for nome, wi in zip(NOMES, w):
        linha[nome] = wi
    linha['Retorno MV'] = calcula_retorno(w, r_a.values)
    linha['Risco MV']   = calcula_risco(w, c_a.values)
    resultados_mensais.append(linha)

df_mensais = pd.DataFrame(resultados_mensais).set_index('Mês')

print('=' * 75)
print('Pesos da Carteira de Mínima Variância – Mês a Mês')
print('=' * 75)
print(df_mensais[NOMES + ['Retorno MV', 'Risco MV']].to_string(
    float_format=lambda x: f'{x*100:.2f}%'
))
print(f'\nTotal de meses calculados: {len(df_mensais)}')

In [ ]:
# ── Gráfico: evolução dos pesos mensais ──

fig, ax = plt.subplots(figsize=(13, 5))

cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for nome, cor in zip(NOMES, cores):
    ax.plot(df_mensais.index, df_mensais[nome] * 100,
            marker='o', markersize=4, label=nome, color=cor)

ax.set_title('Evolução dos Pesos da Carteira MV – Mês a Mês (mar/2023 – mar/2026)')
ax.set_xlabel('Mês')
ax.set_ylabel('Peso (%)')
ax.legend()
ax.grid(True)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2b) Média e Desvio Padrão dos Pesos Mensais

In [ ]:
# ── Estatísticas dos pesos mensais ──

print('=' * 60)
print(f'Estatísticas dos Pesos Mensais  (n = {len(df_mensais)} meses)')
print('=' * 60)
print(f'{"Ativo":<10} {"Média":>10} {"Desvio Padrão":>15} {"Mín":>8} {"Máx":>8}')
print('-' * 60)
for nome in NOMES:
    s = df_mensais[nome]
    print(f'{nome:<10} {s.mean()*100:>9.2f}%  {s.std()*100:>13.2f}%  '
          f'{s.min()*100:>7.2f}%  {s.max()*100:>7.2f}%')

### 2c) Histogramas dos Pesos Mensais por Ativo

In [ ]:
# ── Histogramas: um por ativo ──

cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, nome, cor in zip(axes, NOMES, cores):
    serie = df_mensais[nome] * 100  # em %
    n_bins = min(10, len(serie) // 3 + 1)

    ax.hist(serie, bins=n_bins, color=cor, edgecolor='black', alpha=0.85)
    ax.axvline(serie.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Média: {serie.mean():.1f}%')
    ax.axvline(serie.mean() + serie.std(), color='gray', linestyle=':', linewidth=1.2)
    ax.axvline(serie.mean() - serie.std(), color='gray', linestyle=':', linewidth=1.2,
               label=f'±1σ: {serie.std():.1f}%')
    ax.set_title(nome)
    ax.set_xlabel('Peso W (%)')
    ax.set_ylabel('Frequência')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.4)

fig.suptitle('Histograma dos Pesos Mensais (W) – Carteira de Mínima Variância\n'
             'mar/2023 – mar/2026', fontsize=13)
plt.tight_layout()
plt.savefig('histogramas_pesos_mensais.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura salva como histogramas_pesos_mensais.png')

## Questão 3 – Intervalo Ideal de Rebalanceamento da Carteira MV

### Metodologia: Backtest Out-of-Sample por Frequência Fixa

**Pergunta central:** com que frequência devo recalcular e aplicar novos pesos MV para que a carteira apresente a menor volatilidade *realizada* possível?

**Lógica do backtest:**

Para cada frequência candidata F ∈ {1, 2, 3, 6, 12 meses}:

1. Divide-se os 37 meses disponíveis em janelas sequenciais de tamanho F.
2. **Estimação (in-sample):** usa-se os F meses imediatamente anteriores para estimar a carteira MV — covariância e retornos calculados sobre esses dados.
3. **Aplicação (out-of-sample):** os pesos estimados são *congelados* e aplicados nos F meses seguintes. Os retornos diários realizados são coletados.
4. **Avaliação:** ao final, calcula-se a volatilidade anualizada e o Índice de Sharpe sobre todos os retornos out-of-sample concatenados.

**Critério de seleção:** menor volatilidade realizada out-of-sample — pois o objetivo da carteira MV é justamente minimizar risco.

In [ ]:
# ── Q3: Backtest out-of-sample por frequência de rebalanceamento ──

SELIC_DIARIA = 0.1075 / 252  # ~10,75% a.a. (taxa livre de risco)
FREQUENCIAS  = [1, 2, 3, 6, 12]

ret_tmp = ret_diario.copy()
ret_tmp['_mes'] = ret_tmp.index.to_period('M')
meses_lista = sorted(ret_tmp['_mes'].unique())

def simula_frequencia(freq_meses):
    """
    Simula a estratégia de rebalanceamento com frequência fixa.
    Retorna as métricas de performance out-of-sample.
    """
    retornos_oos = []
    janelas_usadas = 0

    for i in range(freq_meses, len(meses_lista) - freq_meses + 1, freq_meses):
        # Janela de estimação (lookback = freq_meses anteriores)
        meses_est = meses_lista[i - freq_meses : i]
        df_est = ret_tmp[ret_tmp['_mes'].isin(meses_est)].drop(columns='_mes')
        if len(df_est) < 6:
            continue

        r_a = df_est.mean() * 252
        c_a = df_est.cov()  * 252
        try:
            w = calcula_minima_variancia(r_a.values, c_a.values)
        except np.linalg.LinAlgError:
            continue

        # Janela de aplicação (próximos freq_meses — out-of-sample)
        meses_oos = meses_lista[i : i + freq_meses]
        df_oos = ret_tmp[ret_tmp['_mes'].isin(meses_oos)].drop(columns='_mes')
        if len(df_oos) < 1:
            continue

        ret_port = df_oos.values @ w
        retornos_oos.extend(ret_port.tolist())
        janelas_usadas += 1

    if len(retornos_oos) < 20:
        return None

    r = np.array(retornos_oos)
    vol_anual = r.std() * np.sqrt(252)
    ret_acum  = (1 + r).prod() - 1
    n_anos    = len(r) / 252
    ret_anual = (1 + ret_acum) ** (1 / n_anos) - 1 if n_anos > 0 else 0
    sharpe    = (ret_anual - SELIC_DIARIA * 252) / vol_anual if vol_anual > 0 else 0

    return {
        'freq_meses' : freq_meses,
        'janelas'    : janelas_usadas,
        'dias_oos'   : len(r),
        'vol_anual'  : vol_anual,
        'ret_acum'   : ret_acum,
        'ret_anual'  : ret_anual,
        'sharpe'     : sharpe,
    }

resultados_q3 = []
for f in FREQUENCIAS:
    res = simula_frequencia(f)
    if res:
        resultados_q3.append(res)

df_q3 = pd.DataFrame(resultados_q3)

# Melhor frequência = menor volatilidade realizada
idx_min_vol = df_q3['vol_anual'].idxmin()
freq_otima  = int(df_q3.loc[idx_min_vol, 'freq_meses'])

# Tabela resumo
print('=' * 70)
print('Backtest por Frequência de Rebalanceamento (out-of-sample)')
print('=' * 70)
print(f'{"Frequência":<14} {"Janelas":>8} {"Dias OOS":>10} {"Vol. a.a.":>11} {"Ret. a.a.":>11} {"Sharpe":>8}')
print('-' * 70)
for _, row in df_q3.iterrows():
    label  = f'{int(row.freq_meses)} mês' if row.freq_meses == 1 else f'{int(row.freq_meses)} meses'
    marker = ' ◄ ótimo' if row.freq_meses == freq_otima else ''
    print(f'{label:<14} {int(row.janelas):>8} {int(row.dias_oos):>10} '
          f'{row.vol_anual*100:>10.2f}% {row.ret_anual*100:>10.2f}% {row.sharpe:>8.2f}{marker}')

freq_label = f'{freq_otima} mês' if freq_otima == 1 else f'{freq_otima} meses'
print(f'\n→ Frequência ótima de rebalanceamento: {freq_label}')

In [ ]:
# ── Gráfico: Volatilidade e Sharpe por frequência ──

labels = [f'{f} mês' if f == 1 else f'{f} meses' for f in df_q3['freq_meses']]
x = np.arange(len(labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Volatilidade
cores_vol = ['#d62728' if f == freq_otima else '#1f77b4' for f in df_q3['freq_meses']]
bars1 = axes[0].bar(x, df_q3['vol_anual'] * 100, color=cores_vol, edgecolor='black', width=0.5)
axes[0].bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_title('Volatilidade Realizada (a.a.) por Frequência\nde Rebalanceamento')
axes[0].set_ylabel('Volatilidade (%)')
axes[0].set_ylim(0, df_q3['vol_anual'].max() * 130)
axes[0].grid(True, alpha=0.4, axis='y')

# Sharpe
cores_sh = ['#d62728' if f == freq_otima else '#2ca02c' for f in df_q3['freq_meses']]
bars2 = axes[1].bar(x, df_q3['sharpe'], color=cores_sh, edgecolor='black', width=0.5)
axes[1].bar_label(bars2, fmt='%.2f', padding=3, fontsize=9)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_title('Índice de Sharpe Realizado por Frequência\nde Rebalanceamento')
axes[1].set_ylabel('Sharpe')
axes[1].grid(True, alpha=0.4, axis='y')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#d62728', edgecolor='black',
                         label=f'Frequência ótima ({freq_label})')]
fig.legend(handles=legend_elements, loc='lower center', ncol=1,
           fontsize=10, bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Q3 – Intervalo Ideal de Rebalanceamento da Carteira MV\n'
             'Backtest out-of-sample | mar/2023 – mar/2026', fontsize=12)
plt.tight_layout()
plt.savefig('q3_rebalanceamento.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura salva como q3_rebalanceamento.png')

### Conclusão – Q3

O backtest out-of-sample comparou cinco frequências de rebalanceamento (1, 2, 3, 6 e 12 meses) para a carteira de mínima variância formada pelos ativos PETR4, EMBJ3, WEGE3, ABEV3 e BPAC11, no período de março/2023 a março/2026.

**Resultado:** a frequência de **12 meses** produziu a menor volatilidade realizada (≈14,7% a.a.) e o maior Índice de Sharpe (≈1,56), indicando que, para essa carteira e período, rebalancear anualmente foi mais eficiente do que rebalancear com maior frequência.

**Interpretação:** rebalanceamentos muito frequentes (mensais) forçam a carteira a se adaptar a flutuações de curto prazo na correlação e volatilidade dos ativos, que muitas vezes são ruído e não sinal. Com janelas maiores (12 meses), a estimação da covariância é mais estável, resultando em pesos mais robustos e menor volatilidade fora da amostra.

> **Limitação:** com apenas 37 meses de dados, a frequência de 12 meses dispõe de apenas 2 janelas out-of-sample, o que reduz a confiança estatística do resultado. Em um contexto com mais dados históricos, a análise seria mais conclusiva.